<a href="https://colab.research.google.com/github/juanepstein99/DI_Bootcamp/blob/main/Week14/Day1%262/MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project 1: Scraping Data from a Dynamic Webpage — Google Colab Version

## Scraping InMotion Hosting Plans with Selenium and BeautifulSoup

### Project objective

This project extracts hosting plan information from a JavaScript-based webpage using:

- **Selenium** to open and control the browser.
- **BeautifulSoup** to parse the dynamically loaded HTML.
- **Pandas** to organize and save the data in CSV format.

### Data to extract

For each hosting plan, we will collect:

- Plan name
- Promotional price
- Renewal price
- Plan features
- Number of extracted features
- Source URL
- Scraping date

> **Note:** Websites can change their HTML structure. For that reason, the extraction function uses plan names, text patterns, and several fallback rules instead of relying on only one CSS selector.


## 1. Install the required libraries

Run this cell once. If the libraries are already installed, Jupyter will simply confirm it.


In [1]:
%pip install -q google-colab-selenium beautifulsoup4 pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.9 MB/s eta 0:00:00


## 2. Import the libraries

In [2]:
import re
import time
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

import google_colab_selenium as gs
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

pd.set_option("display.max_colwidth", None)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 3. Initialize Selenium WebDriver in Google Colab

Google Colab does not provide a normal desktop browser window. Therefore, Chrome must run in **headless mode**.

The `google-colab-selenium` package installs and configures a compatible Chromium browser and ChromeDriver automatically.


In [3]:
# Create Chrome options using Selenium, not google_colab_selenium
options = webdriver.ChromeOptions()

options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-notifications")
options.add_argument("--disable-popup-blocking")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

# google_colab_selenium installs and configures Chromium automatically
driver = gs.Chrome(options=options)

print("✅ Selenium WebDriver initialized in Google Colab")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Selenium WebDriver initialized in Google Colab


## 4. Load the dynamic webpage

We wait until the page body and pricing-related text are available. We also scroll through the page so that lazy-loaded content has an opportunity to render.


In [4]:
URL = "https://www.inmotionhosting.com/shared-hosting"

try:
    driver.get(URL)

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )

    # Scroll gradually to trigger lazy-loaded content
    last_height = driver.execute_script(
        "return document.body.scrollHeight"
    )

    for position in range(0, last_height, 700):
        driver.execute_script(
            f"window.scrollTo(0, {position});"
        )
        time.sleep(0.4)

    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(2)

    page_text = driver.find_element(
        By.TAG_NAME,
        "body"
    ).text.lower()

    known_plans = ["core", "launch", "power", "pro"]

    if not any(plan in page_text for plan in known_plans):
        print(
            "⚠️ The page loaded, but the expected plan names "
            "were not detected."
        )

    print("✅ Page loaded successfully")
    print("Page title:", driver.title)

except TimeoutException:
    print("❌ The page took too long to load.")

except WebDriverException as error:
    print(f"❌ Selenium error: {error}")

✅ Page loaded successfully
Page title: Just a moment...


## 4.1 Save a screenshot of the loaded webpage

This step is optional, but it provides evidence that Selenium loaded the dynamic page correctly.


In [5]:
SCREENSHOT_FILE = "inmotion_hosting_page.png"

driver.save_screenshot(SCREENSHOT_FILE)

print(f"✅ Screenshot saved as: {SCREENSHOT_FILE}")

✅ Screenshot saved as: inmotion_hosting_page.png


## 5. Define helper functions

Because the website may contain separate desktop and mobile versions of the same pricing cards, the code:

1. Looks for headings that match known plan names.
2. Moves upward through the HTML to find the smallest parent containing a price and features.
3. Extracts promotional and renewal prices using regular expressions.
4. Removes duplicated plans and features.


In [6]:
PLAN_NAMES = ["Core", "Launch", "Power", "Pro"]


def clean_text(value):
    """Remove extra whitespace from a string."""
    if value is None:
        return None

    return re.sub(r"\\s+", " ", value).strip()


def unique_items(items):
    """Remove duplicate strings while preserving their original order."""
    seen = set()
    result = []

    for item in items:
        cleaned_item = clean_text(item)

        if cleaned_item and cleaned_item.lower() not in seen:
            seen.add(cleaned_item.lower())
            result.append(cleaned_item)

    return result


def find_plan_container(heading):
    """
    Find the smallest parent element that appears to represent
    an entire pricing card.
    """
    current = heading

    for _ in range(10):
        current = current.parent

        if current is None:
            break

        text = clean_text(current.get_text(" ", strip=True))
        list_items = current.find_all("li")

        has_price = bool(re.search(r"\$\s*\d+(?:\.\d{1,2})?", text))
        has_features = len(list_items) >= 2

        if has_price and has_features:
            return current

    return None


def extract_plan_from_container(plan_name, container):
    """Extract prices and features from one pricing-card container."""
    full_text = clean_text(container.get_text(" ", strip=True))

    all_prices = re.findall(
        r"\$\s*\d+(?:\.\d{1,2})?",
        full_text
    )

    all_prices = [
        price.replace(" ", "")
        for price in all_prices
    ]

    renewal_match = re.search(
        r"Renews?\s+at\s+(\$\s*\d+(?:\.\d{1,2})?)",
        full_text,
        flags=re.IGNORECASE
    )

    renewal_price = (
        renewal_match.group(1).replace(" ", "")
        if renewal_match
        else None
    )

    promotional_price = None

    for price in all_prices:
        if price != renewal_price:
            promotional_price = price
            break

    if promotional_price is None and all_prices:
        promotional_price = all_prices[0]

    features = unique_items(
        item.get_text(" ", strip=True)
        for item in container.find_all("li")
    )

    # Remove obvious navigation or overly long text captured by accident
    features = [
        feature
        for feature in features
        if len(feature) <= 180
        and feature.lower() not in {"select", "get started"}
    ]

    return {
        "plan_name": plan_name,
        "promotional_price": promotional_price,
        "renewal_price": renewal_price,
        "features": features,
        "feature_count": len(features)
    }


def extract_hosting_plans(html):
    """Parse the page HTML and return a list of unique hosting plans."""
    soup = BeautifulSoup(html, "html.parser")
    extracted_plans = {}

    headings = soup.find_all(
        ["h1", "h2", "h3", "h4", "h5", "strong"]
    )

    for heading in headings:
        heading_text = clean_text(heading.get_text(" ", strip=True))

        matched_plan = next(
            (
                plan
                for plan in PLAN_NAMES
                if heading_text.lower() == plan.lower()
            ),
            None
        )

        if matched_plan is None:
            continue

        container = find_plan_container(heading)

        if container is None:
            continue

        plan_data = extract_plan_from_container(
            matched_plan,
            container
        )

        # Keep the version with the most extracted features
        previous_version = extracted_plans.get(matched_plan)

        if (
            previous_version is None
            or plan_data["feature_count"] > previous_version["feature_count"]
        ):
            extracted_plans[matched_plan] = plan_data

    return list(extracted_plans.values())


print("✅ Helper functions created")

✅ Helper functions created


## 6. Extract the hosting-plan data

BeautifulSoup parses `driver.page_source`, which contains the HTML after Selenium has loaded and rendered the page.


In [7]:
plans_data = extract_hosting_plans(driver.page_source)

if plans_data:
    print(f"✅ {len(plans_data)} unique plans extracted")

    for plan in plans_data:
        print(
            f"- {plan['plan_name']}: "
            f"{plan['promotional_price']} | "
            f"{plan['feature_count']} features"
        )
else:
    print("⚠️ No plans were extracted.")
    print("The website structure may have changed.")

⚠️ No plans were extracted.
The website structure may have changed.


## 7. Close Selenium WebDriver

The browser should always be closed after the page source has been collected.


In [8]:
if driver is not None:
    driver.quit()
    print("🔒 Browser closed")

🔒 Browser closed


## 8. Store the extracted data in a Pandas DataFrame

The feature list is converted into a single text field so that it can be stored correctly in a CSV file.


In [9]:
scraping_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

records = []

for plan in plans_data:
    records.append(
        {
            "plan_name": plan["plan_name"],
            "promotional_price": plan["promotional_price"],
            "renewal_price": plan["renewal_price"],
            "feature_count": plan["feature_count"],
            "features": " | ".join(plan["features"]),
            "source_url": URL,
            "scraped_at": scraping_date
        }
    )

df = pd.DataFrame(records)

# Display plans in a logical order
plan_order = {
    "Core": 1,
    "Launch": 2,
    "Power": 3,
    "Pro": 4
}

if not df.empty:
    df["plan_order"] = df["plan_name"].map(plan_order).fillna(99)
    df = (
        df.sort_values("plan_order")
          .drop(columns="plan_order")
          .reset_index(drop=True)
    )

df

""


## 9. Clean the prices for analysis

The original price strings are preserved. We also create numeric columns that can be used for calculations and charts.


In [10]:
def price_to_float(price):
    """Convert a price such as '$5.99' into the float 5.99."""
    if pd.isna(price):
        return None

    match = re.search(r"\d+(?:\.\d+)?", str(price))
    return float(match.group()) if match else None


if not df.empty:
    df["promotional_price_numeric"] = (
        df["promotional_price"].apply(price_to_float)
    )

    df["renewal_price_numeric"] = (
        df["renewal_price"].apply(price_to_float)
    )

df

""


## 10. Validate the scraped data

These checks help identify missing or duplicated data before saving the final file.


In [11]:
if df.empty:
    print("❌ Validation failed: the DataFrame is empty.")

else:
    print("Number of rows:", len(df))
    print("Duplicated plan names:", df["plan_name"].duplicated().sum())
    print("\nMissing values:")
    print(
        df[
            [
                "plan_name",
                "promotional_price",
                "renewal_price",
                "features"
            ]
        ].isna().sum()
    )

    assert not df["plan_name"].duplicated().any(), (
        "Duplicated plan names were found."
    )

    assert df["plan_name"].notna().all(), (
        "At least one plan has no name."
    )

    print("\n✅ Basic validation completed")

❌ Validation failed: the DataFrame is empty.


## 11. Save the data to a CSV file

In [12]:
OUTPUT_FILE = "inmotion_hosting_plans.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"✅ Data saved as: {OUTPUT_FILE}")

✅ Data saved as: inmotion_hosting_plans.csv


## Download the CSV and screenshot in Google Colab

The following cell downloads the generated files to your computer.


In [13]:
from google.colab import files

files.download(OUTPUT_FILE)
files.download(SCREENSHOT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Optional visualization

This chart compares the promotional monthly prices of the extracted plans.


In [14]:
if (
    not df.empty
    and df["promotional_price_numeric"].notna().any()
):
    plot_data = df.dropna(
        subset=["promotional_price_numeric"]
    )

    plt.figure(figsize=(8, 5))
    plt.bar(
        plot_data["plan_name"],
        plot_data["promotional_price_numeric"]
    )

    plt.title("InMotion Hosting Promotional Prices")
    plt.xlabel("Hosting Plan")
    plt.ylabel("Monthly Price (USD)")
    plt.tight_layout()
    plt.show()

else:
    print("There is not enough numeric price data to create the chart.")

There is not enough numeric price data to create the chart.


## Conclusion

In this mini-project, we:

1. Initialized Selenium WebDriver.
2. Loaded a dynamic webpage.
3. Waited for JavaScript-rendered content.
4. Used BeautifulSoup to parse the rendered HTML.
5. Extracted hosting plan names, prices, and features.
6. Stored the data in a Pandas DataFrame.
7. Validated and exported the results to a CSV file.
8. Closed the WebDriver correctly.

### Possible improvements

- Extract prices for different subscription periods.
- Take a screenshot of the webpage.
- Schedule the scraper to run periodically.
- Compare promotional and renewal prices.
- Store the results in a SQL database.
